# Prepare ML Training Data - Multi-Route

This notebook generalizes the single-route pipeline to handle any combination of cities.

**Input:**
- List of cities (must have corresponding weather feature files)
- Processed weather features from BOM weather data
- Raw BITRE flight data

**Output:**
- `ml_training_data_multiroute.csv` - Combined flight + weather data for all route pairs


## 1. User Input

Define the cities to include. Each city must have:
1. A weather features file: `data/processed/features_{city_code}.csv`
2. Flight routes in the BITRE data using the city's full name

First, create a mapping between cities' full name and code.

In [ ]:
# =============================================================================
# USER_INPUT - Modify this section to add/remove cities
# =============================================================================

# City mapping: full name (as in BITRE data) -> code (as in weather files)
# Add new cities here as needed
CITY_MAPPING = {
    'Sydney': 'syd',
    'Melbourne': 'mel',
    'Hobart': 'hba',
    'Brisbane': 'bne',
    # Add more cities as weather data becomes available:
    # 'Perth': 'per',
    # 'Adelaide': 'adl',
}

# Select which cities to include in this run
# Must be a subset of CITY_MAPPING keys
CITIES_TO_INCLUDE = ['Sydney', 'Melbourne', 'Hobart']

# Output filename
OUTPUT_FILENAME = 'ml_training_data_multiroute.csv'

print(f"Cities to include: {CITIES_TO_INCLUDE}")
print(f"City codes: {[CITY_MAPPING[c] for c in CITIES_TO_INCLUDE]}")

## 2. Setup

In [ ]:
import pandas as pd
import numpy as np
import glob
import os
from itertools import permutations
from datetime import datetime
from dateutil.relativedelta import relativedelta
import requests

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

print("Setup complete")

To create the routes from the list of included cities, generate **permutations** of two cities (one pair). Permutation is used here because the order of the pair (i.e. direction) matters.

In [ ]:
# Generate all route pairs (permutations, not combinations - direction matters)
route_pairs = list(permutations(CITIES_TO_INCLUDE, 2)) 

print(f"Route pairs to process ({len(route_pairs)} routes):")
for dep, arr in route_pairs:
    print(f"  {dep} -> {arr}")

In [ ]:
# Verify weather feature files exist for all cities
print("Checking weather feature files...")
missing_files = []

for city in CITIES_TO_INCLUDE:
    code = CITY_MAPPING[city]
    filepath = f"../data/processed/features_{code}.csv"
    if os.path.exists(filepath):
        print(f"  \u2713 {city} ({code}): {filepath}")
    else:
        print(f"  \u2717 {city} ({code}): FILE NOT FOUND - {filepath}")
        missing_files.append(filepath)

if missing_files:
    raise FileNotFoundError(f"Missing weather files: {missing_files}")
else:
    print("\nAll weather feature files found!")

## 3. Load and Clean Flight Data

Download the latest raw BITRE data and filter to selected routes.

The logic for the download function is:
- First, try the month before the current date
- If not available, try the month before that
- If also not available or no access to the website, try the latest locally stored data

In [ ]:
def download_bitre_data(save_path="../data/raw/"):
    """
    Download the latest BITRE flight data by trying recent months.
    """
    base_url = "https://www.bitre.gov.au/sites/default/files/documents/"
    current_date = datetime.now()
    
    # Important to use headers, otherwise blocked by BITRE
    headers = {
        'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36'
    }
    
    for months_back in range(1, 3):
        target_date = current_date - relativedelta(months=months_back)
        month_name = target_date.strftime("%B").lower()     # Make month name lower case to match BITRE
        year = target_date.year
        
        filename = f"OTP_Time_Series_Master_Current_{month_name}_{year}.xlsx"
        url = base_url + filename
        
        print(f"Attempting to download: {filename}")
        
        # Use try/except here so the function does not break in the case of request error 
        try:
            response = requests.get(url, headers=headers, timeout=60)
            if response.status_code == 200:     # Status code 200 means successful request
                filepath = save_path + filename
                with open(filepath, 'wb') as f:
                    f.write(response.content)
                print(f"\u2713 Successfully downloaded: {filename}")
                return filepath
            else:
                print(f"  Not found (HTTP {response.status_code})")
        except requests.RequestException as e:
            print(f"  Failed: {type(e).__name__}")
    
    print("\n\u26a0 Automatic download failed. Will use existing local file.")
    return None

# Attempt to download latest data
downloaded_file = download_bitre_data()
print(f"\n{'='*80}")

In [ ]:
# Load BITRE data
if downloaded_file:
    data_file = downloaded_file
else:
    local_files = glob.glob("../data/raw/OTP_Time_Series_Master_Current_*.xlsx")
    if local_files:
        data_file = max(local_files)
        print(f"Using existing local file: {data_file}")
    else:
        raise FileNotFoundError("No BITRE data file found.")

# Load all sheets
all_sheets_temp = pd.read_excel(data_file, sheet_name=None)
df_raw = pd.concat(all_sheets_temp.values(), ignore_index=True)

# Remove duplicates
rows_before = len(df_raw)
df_raw = df_raw.drop_duplicates()
rows_dropped = rows_before - len(df_raw)
if rows_dropped > 0:
    print(f"Removed {rows_dropped} duplicate rows")

# Convert date
df_raw['Month'] = pd.to_datetime(df_raw['Month'], format='%m/%Y')
df_raw['year_month'] = df_raw['Month'].dt.to_period('M').astype(str)


Use the city-pair permutations generated earlier to filter BITRE data and only retain the relevant routes. Since airline identity is used as a model feature, the "All Airlines" aggregate rows are removed.

In [ ]:
# Build route filter: all permutations of selected cities
route_names = []
for dep, arr in route_pairs:
    route_names.append(f"{dep}-{arr}")

print(f"Filtering for routes: {route_names}")

# Filter to selected routes and exclude "All Airlines" aggregate
df_flights = df_raw[
    (df_raw['Route'].isin(route_names)) &
    (df_raw['Airline'] != 'All Airlines')
].copy()

print(f"\nFiltered to {len(df_flights)} records")
print(f"\nRoutes found:")
print(df_flights['Route'].value_counts())

In [ ]:
# Data cleaning steps
print("Cleaning flight data...")
print(f"\n{'='*80}")

# 1. Remove rows with Sectors Flown = 0
rows_before = len(df_flights)
df_flights = df_flights[df_flights['Sectors Flown'] > 0].copy()
print(f"Removed {rows_before - len(df_flights)} rows with Sectors Flown = 0")

# 2. Calculate target variables
df_flights['delay_rate'] = (df_flights['Sectors Flown'] - df_flights['Arrivals On Time']) / df_flights['Sectors Flown']
df_flights['is_high_delay'] = (df_flights['delay_rate'] > 0.25).astype(int)

# 3. Extract year
df_flights['year'] = df_flights['Month'].dt.year

# 4. Exclude COVID period (April-December 2020)
covid_mask = (df_flights['Month'] >= '2020-04-01') & (df_flights['Month'] <= '2020-12-31')
rows_before = len(df_flights)
df_flights = df_flights[~covid_mask].copy()
print(f"Removed {rows_before - len(df_flights)} rows from COVID period (Apr-Dec 2020)")

# 5. Rename columns
column_mapping = {
    'Route': 'route',
    'Departing Port': 'departing_port',
    'Arriving Port': 'arriving_port',
    'Airline': 'airline',
    'Month': 'month',
    'Sectors Scheduled': 'sectors_scheduled',
    'Sectors Flown': 'sectors_flown',
    'Cancellations': 'cancellations',
    'Arrivals On Time': 'arrivals_on_time',
    'Arrivals Delayed': 'arrivals_delayed',
    'Cancellations \n\n(%)': 'cancellations_pct',
}
df_flights = df_flights.rename(columns=column_mapping)

print(f"\nFinal flight data: {len(df_flights)} records")
print(f"Date range: {df_flights['year_month'].min()} to {df_flights['year_month'].max()}")

In [ ]:
# Summary by route
print("Records per route:")
print(df_flights.groupby(['departing_port', 'arriving_port']).size().reset_index(name='count'))
print(f"\n{'='*80}")

print("\nAirlines per route:")
for (dep, arr), group in df_flights.groupby(['departing_port', 'arriving_port']):
    airlines = group['airline'].unique()
    print(f"  {dep} -> {arr}: {len(airlines)} airlines")

## 4. Load Weather Features

Load weather feature files for all cities.

In [ ]:
# Load weather features for each city
weather_data = {}

for city in CITIES_TO_INCLUDE:
    code = CITY_MAPPING[city]
    filepath = f"../data/processed/features_{code}.csv"
    
    df_weather = pd.read_csv(filepath)
    weather_data[city] = df_weather
    
    print(f"{city} ({code}): {len(df_weather)} months, {len(df_weather.columns)-1} features")
    print(f"  Date range: {df_weather['year_month'].min()} to {df_weather['year_month'].max()}")

print(f"\n{'='*80}")
print(f"Loaded weather data for {len(weather_data)} cities")

In [ ]:
# Get weather feature column names (same across all cities)
sample_city = CITIES_TO_INCLUDE[0]
weather_feature_cols = [col for col in weather_data[sample_city].columns if col != 'year_month']

print(f"Weather feature columns ({len(weather_feature_cols)}):")
for col in weather_feature_cols:
    print(f"  - {col}")

## 5. Merge Flight and Weather Data

For each flight record, merge weather data from both departure and arrival airports.

In [ ]:
def prepare_weather_with_suffix(df_weather, suffix):
    """
    Prepare weather dataframe with column suffix (_dep or _arr).
    """
    df = df_weather.copy()
    rename_dict = {col: f"{col}{suffix}" for col in df.columns if col != 'year_month'}
    df = df.rename(columns=rename_dict)
    return df

# Prepare departure and arrival weather dataframes for each city
weather_dep = {city: prepare_weather_with_suffix(weather_data[city], '_dep') for city in CITIES_TO_INCLUDE}
weather_arr = {city: prepare_weather_with_suffix(weather_data[city], '_arr') for city in CITIES_TO_INCLUDE}

print("Weather dataframes prepared with _dep and _arr suffixes")
print(f"Example columns for {CITIES_TO_INCLUDE[0]} departure: {weather_dep[CITIES_TO_INCLUDE[0]].columns[:5].tolist()}...")

In [ ]:
# Merge weather data based on departing and arriving ports
print("Merging weather data...")
print(f"\n{'='*80}")

merged_dfs = []

for (dep_city, arr_city) in route_pairs:
    # Filter flights for this route
    route_mask = (df_flights['departing_port'] == dep_city) & (df_flights['arriving_port'] == arr_city)
    df_route = df_flights[route_mask].copy()
    
    if len(df_route) == 0:
        print(f"  {dep_city} -> {arr_city}: No flights found, skipping")
        continue
    
    # Merge departure weather
    df_route = df_route.merge(weather_dep[dep_city], on='year_month', how='left')
    
    # Merge arrival weather
    df_route = df_route.merge(weather_arr[arr_city], on='year_month', how='left')
    
    merged_dfs.append(df_route)
    
    # Count nulls
    null_dep = df_route[[c for c in df_route.columns if c.endswith('_dep')]].isnull().sum().sum()
    null_arr = df_route[[c for c in df_route.columns if c.endswith('_arr')]].isnull().sum().sum()
    
    print(f"  {dep_city} -> {arr_city}: {len(df_route)} rows (null dep: {null_dep}, null arr: {null_arr})")

# Combine all routes
df_combined = pd.concat(merged_dfs, ignore_index=True)
print(f"\n{'='*80}")
print(f"Combined dataset: {len(df_combined)} rows")

In [ ]:
# Check for null values
null_counts = df_combined.isnull().sum()
cols_with_nulls = null_counts[null_counts > 0]

if len(cols_with_nulls) > 0:
    print("Columns with null values after merge:")
    print(cols_with_nulls)
    print(f"\n{'='*80}")
    
    # Show which year_months have missing data
    null_rows = df_combined[df_combined.isnull().any(axis=1)]
    print(f"Rows with null values: {len(null_rows)}")
    print(f"Year months with missing data: {sorted(null_rows['year_month'].unique())[:10]}...")
else:
    print("No null values found after merge")

## 6. Final Dataset Preparation

In [ ]:
# Define column order
flight_cols = ['departing_port', 'arriving_port', 'airline', 'month', 'year_month', 'year',
               'sectors_scheduled', 'sectors_flown', 'arrivals_on_time', 'arrivals_delayed',
               'cancellations', 'cancellations_pct']

target_cols = ['delay_rate', 'is_high_delay']

weather_dep_cols = sorted([c for c in df_combined.columns if c.endswith('_dep')])
weather_arr_cols = sorted([c for c in df_combined.columns if c.endswith('_arr')])

# Select only columns that exist
flight_cols = [c for c in flight_cols if c in df_combined.columns]

print(f"Flight metadata columns: {len(flight_cols)}")
print(f"Target columns: {len(target_cols)}")
print(f"Departure weather features: {len(weather_dep_cols)}")
print(f"Arrival weather features: {len(weather_arr_cols)}")

# Reorder columns
column_order = flight_cols + target_cols + weather_dep_cols + weather_arr_cols
df_final = df_combined[column_order].copy()

# Sort by date, airline, and route
df_final = df_final.sort_values(['year_month', 'airline', 'departing_port', 'arriving_port']).reset_index(drop=True)

print(f"\nFinal dataset shape: {df_final.shape}")

In [ ]:
# Summary statistics
print("Dataset Summary")
print(f"{'='*80}")
print(f"\nTotal records: {len(df_final)}")
print(f"Date range: {df_final['year_month'].min()} to {df_final['year_month'].max()}")
print(f"\nRoutes:")
print(df_final.groupby(['departing_port', 'arriving_port']).size().reset_index(name='count'))

print(f"\nAirlines: {df_final['airline'].nunique()}")
print(df_final['airline'].value_counts())

print(f"\n{'='*80}")
print("\nTarget variable summary:")
print(f"\ndelay_rate:")
print(df_final['delay_rate'].describe())
print(f"\nis_high_delay distribution:")
print(df_final['is_high_delay'].value_counts())

In [ ]:
# Preview final dataset
print("Preview of final dataset:")
df_final.head(10)

## 7. Export

In [ ]:
# Save to CSV
output_path = f"../data/processed/{OUTPUT_FILENAME}"
df_final.to_csv(output_path, index=False)

print(f"ML training data saved to: {output_path}")
print(f"File size: {os.path.getsize(output_path) / 1024:.1f} KB")

## 8. Summary

**Data Combination Steps Completed:**

1. Configured cities and generated all route pair permutations
2. Loaded and cleaned raw BITRE flight data
3. Filtered to selected routes, excluded COVID period
4. Loaded weather features for all cities
5. Merged weather data based on flight direction:
   - Departure airport weather (suffix: `_dep`)
   - Arrival airport weather (suffix: `_arr`)
6. Combined all routes into single dataset
7. Saved final ML training data

**To add more cities:**
1. Ensure weather feature file exists: `data/processed/features_{city_code}.csv`
2. Add city to `CITY_MAPPING` dictionary
3. Add city to `CITIES_TO_INCLUDE` list
4. Re-run notebook